In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from collections import  Counter
import SimpleITK as sitk
import nibabel as nib
import time

In [2]:
def sitk_new_blank_image(size, spacing, direction, origin, default_value=0.):
    image = sitk.GetImageFromArray(np.ones(size, dtype=np.float).T * default_value)
    image.SetSpacing(spacing)
    image.SetDirection(direction)
    image.SetOrigin(origin)
    return image

def sitk_resample_to_image(image, reference_image, interpolator, default_value=0., transform=None,
                           output_pixel_type=None):
    if transform is None:
        transform = sitk.Transform()
        transform.SetIdentity()
    if output_pixel_type is None:
        output_pixel_type = image.GetPixelID()
    resample_filter = sitk.ResampleImageFilter()
    resample_filter.SetInterpolator(interpolator)
    resample_filter.SetTransform(transform)
    resample_filter.SetOutputPixelType(output_pixel_type)
    resample_filter.SetDefaultPixelValue(default_value)
    resample_filter.SetReferenceImage(reference_image)
    return resample_filter.Execute(image)

def generate_data_propotional(series_uid, scale, interpolator, output_pixel_type, is_dcm=True):
    if is_dcm:
        reader = sitk.ImageSeriesReader()
        filenames = reader.GetGDCMSeriesFileNames(series_uid)
        reader.SetFileNames(filenames)
        im = reader.Execute()
    else:
        im = sitk.ReadImage(series_uid)
    ori_spacing = im.GetSpacing()
    ori_size = im.GetSize()
    new_size = [0,0,0]
    for i in range(3):
        new_size[i] = int(ori_spacing[i]*ori_size[i]/(ori_spacing[0]))
    new_spacing = [ori_spacing[0]]*3

#     interpolator = sitk.sitkNearestNeighbor

    black_im = sitk_new_blank_image(new_size, new_spacing, im.GetDirection(), im.GetOrigin())
    new_im = sitk_resample_to_image(im, black_im, interpolator, default_value=0, output_pixel_type=output_pixel_type)

    return new_im

In [3]:
def process_one_case(volume_file, mask_file, outdir, sub_root):
    beg = time.time()
    print(mask_file)
    print('====> begin to process {}'.format(mask_file))
    volume_data = generate_data_propotional(volume_file, None, sitk.sitkLinear, sitk.sitkInt16)
    mask_data = generate_data_propotional(mask_file, None, sitk.sitkNearestNeighbor, sitk.sitkUInt8)
    print('\ttime elpased when rescale data:{:.3f}'.format(time.time()-beg))
    
    basename = os.path.basename(sub_root)
    out_name_volume = os.path.join(outdir, '{}_volume.nii.gz'.format(basename))
    out_name_mask = os.path.join(outdir, '{}_mask.nii.gz'.format(basename))
    
    sitk.WriteImage(volume_data, out_name_volume)
    sitk.WriteImage(mask_data, out_name_mask)
    print('\ttime elpased when save to nii:{:.3f}'.format(time.time()-beg))
    
    out_name_volume_npy = os.path.join(outdir, '{}_volume.npy'.format(basename))
    out_name_mask_npy = os.path.join(outdir, '{}_mask.npy'.format(basename))
    
    ori_volume_data = sitk.GetArrayFromImage(volume_data)
    ori_mask = sitk.GetArrayFromImage(mask_data)
    
    ori_mask[ori_mask > 1] = 1
    
    with open(out_name_volume_npy, 'wb') as f:
        np.save(f, ori_volume_data)
    with open(out_name_mask_npy, 'wb') as f:
        np.save(f, ori_mask)
    
    print('\ttime elpased when save npy:{:.3f}'.format(time.time()-beg))
    
    mix_min = np.min(ori_volume_data)
    mix_max = np.max(ori_volume_data)
    
    print('\ttime elpased when calculate min&max:{:.3f}'.format(time.time()-beg))
    
    
    
    out_name_volume_npy = os.path.join(outdir, '{}_volume_[0-1].npy'.format(basename))
    ori_mix_data_0_1 = (ori_volume_data-mix_min)/(mix_max-mix_min)
    
    with open(out_name_volume_npy, 'wb') as f:
        np.save(f, ori_mix_data_0_1)
    
    print('\ttime elpased when save normalize [0,1] npy:{:.3f}'.format(time.time()-beg))
    
    
    
    out_name_volume_npy = os.path.join(outdir, '{}_volume_[-1-1].npy'.format(basename))
    tmp_v = mix_max+mix_min
    ori_mix_data_0_1 = (ori_volume_data*2-tmp_v)/tmp_v
    
    with open(out_name_volume_npy, 'wb') as f:
        np.save(f, ori_mix_data_0_1)
    print('\ttime elpased when save normalize [-1,1] npy:{:.3f}'.format(time.time()-beg))
    
    
    print('====> end to process {}, Time Elapsed:{:.3f}s\n'.format(volume_file, time.time()-beg))

In [4]:
data_in_root = '../data/Liver'
data_out_root = '../data/processed_liver_liver/scalex1_liver'
os.makedirs(data_out_root, exist_ok=True)
# 生成文件，需要重新生成的时候，请解开注释
for sub_root_name in os.listdir(data_in_root):
    sub_root = os.path.join(data_in_root, sub_root_name)
    if not os.path.isdir(sub_root):
        continue
    if '3Dircadb' not in sub_root:
        continue
    mask_in_series = os.path.join(sub_root, 'MASKS_DICOM/MASKS_DICOM')
    if not os.path.isdir(mask_in_series):
        print('mask label not exist:\t{}'.format(mask_in_series))
        continue
    liver_series_path = os.path.join(mask_in_series, 'liver')
#     print(liver_series_path)
    if not os.path.isdir(liver_series_path):
        continue
    patient_in_series = os.path.join(sub_root, 'PATIENT_DICOM/PATIENT_DICOM')
    if not os.path.isdir(patient_in_series):
        continue
    print(data_out_root)
    process_one_case(patient_in_series, liver_series_path, data_out_root, sub_root)

../data/processed_liver_liver/scalex1_liver
../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM/liver
====> begin to process ../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM/liver
	time elpased when rescale data:26.224
	time elpased when save to nii:36.524
	time elpased when save npy:37.674
	time elpased when calculate min&max:37.794
	time elpased when save normalize [0,1] npy:40.889
	time elpased when save normalize [-1,1] npy:43.998
====> end to process ../data/Liver/3Dircadb1.10/PATIENT_DICOM/PATIENT_DICOM, Time Elapsed:43.998s

../data/processed_liver_liver/scalex1_liver
../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/liver
====> begin to process ../data/Liver/3Dircadb1.20/MASKS_DICOM/MASKS_DICOM/liver
	time elpased when rescale data:52.639
	time elpased when save to nii:76.109
	time elpased when save npy:78.844
	time elpased when calculate min&max:79.085
	time elpased when save normalize [0,1] npy:85.470


/home/zhangwd/.conda/envs/py36/lib/python3.6/site-packages/ipykernel_launcher.py:51: RuntimeWarning: divide by zero encountered in true_divide
/home/zhangwd/.conda/envs/py36/lib/python3.6/site-packages/ipykernel_launcher.py:51: RuntimeWarning: invalid value encountered in true_divide


	time elpased when save normalize [-1,1] npy:91.742
====> end to process ../data/Liver/3Dircadb1.20/PATIENT_DICOM/PATIENT_DICOM, Time Elapsed:91.742s

../data/processed_liver_liver/scalex1_liver
../data/Liver/3Dircadb1.2/MASKS_DICOM/MASKS_DICOM/liver
====> begin to process ../data/Liver/3Dircadb1.2/MASKS_DICOM/MASKS_DICOM/liver
	time elpased when rescale data:30.992
	time elpased when save to nii:44.034
	time elpased when save npy:45.516
	time elpased when calculate min&max:45.668
	time elpased when save normalize [0,1] npy:49.734
	time elpased when save normalize [-1,1] npy:54.002
====> end to process ../data/Liver/3Dircadb1.2/PATIENT_DICOM/PATIENT_DICOM, Time Elapsed:54.002s

../data/processed_liver_liver/scalex1_liver
../data/Liver/3Dircadb1.9/MASKS_DICOM/MASKS_DICOM/liver
====> begin to process ../data/Liver/3Dircadb1.9/MASKS_DICOM/MASKS_DICOM/liver
	time elpased when rescale data:21.959
	time elpased when save to nii:33.726
	time elpased when save npy:34.867
	time elpased when cal

	time elpased when rescale data:42.349
	time elpased when save to nii:55.565
	time elpased when save npy:57.616
	time elpased when calculate min&max:57.771
	time elpased when save normalize [0,1] npy:62.952
	time elpased when save normalize [-1,1] npy:65.908
====> end to process ../data/Liver/3Dircadb1.16/PATIENT_DICOM/PATIENT_DICOM, Time Elapsed:65.908s

../data/processed_liver_liver/scalex1_liver
../data/Liver/3Dircadb1.13/MASKS_DICOM/MASKS_DICOM/liver
====> begin to process ../data/Liver/3Dircadb1.13/MASKS_DICOM/MASKS_DICOM/liver
	time elpased when rescale data:30.720
	time elpased when save to nii:40.466
	time elpased when save npy:42.218
	time elpased when calculate min&max:42.344
	time elpased when save normalize [0,1] npy:46.692
	time elpased when save normalize [-1,1] npy:51.061
====> end to process ../data/Liver/3Dircadb1.13/PATIENT_DICOM/PATIENT_DICOM, Time Elapsed:51.061s

../data/processed_liver_liver/scalex1_liver
../data/Liver/3Dircadb1.12/MASKS_DICOM/MASKS_DICOM/liver
==